# Post-hoc

Scores trained checkpoints. No training. Everything runs through
`scripts/posthoc.py`; each model is rebuilt from the config inside its own
checkpoint, so its resolution and encoding are reproduced exactly.

Order: **inspect → validation → TTA/thresholds → test, once.**

## 0. Colab web UI only — clone

Skip if `/content/fdl-project` exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. Inspect

What each checkpoint actually is — run name, model, resolution, epoch, best metric. No GPU.

Use it to resolve duplicate Drive folders: a re-run lands beside the original as
`<name> (1)`, and `best_epoch` / `best_metric` say which is which.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

CHECKPOINTS_DIR = CHECKPOINTS if HAS_DRIVE else REPO / "trained-models/checkpoints"
OUTPUT_ROOT = REPO / "output/posthoc"


def posthoc(*args: str) -> int:
    command = [sys.executable, str(REPO / "scripts/posthoc.py"),
               "--checkpoints-dir", str(CHECKPOINTS_DIR),
               "--output-root", str(OUTPUT_ROOT), *args]
    print(" ".join(command), "\n")
    return subprocess.run(command, check=False).returncode


posthoc("--inspect")

## 3. Validation

Edit `ONLY` to choose runs. Results append to `comparison.csv` after each model.

In [ ]:
# Pick the runs to score. Substring match on the run name; drop it for all of them.
ONLY = [
    "resnet34_finetune",
    "convnext_tiny_finetune",
    "vit_b_32_finetune",
    "dilated",
    "convnext_big_128",
]

flags = [flag for name in ONLY for flag in ("--only", name)]
posthoc("--split", "validation", *flags)

## 4. TTA and per-class thresholds

Both are free gains if they hold — no retraining. Compare against section 3.

In [ ]:
# TTA averages over the 8 square symmetries -- 8x the inference, no retraining.
# Thresholds are fitted on validation here and reused on test in the next cell.
posthoc("--split", "validation", "--tta", "--tune-thresholds",
        "--results", str(OUTPUT_ROOT / "comparison_tta.csv"), *flags)

## 5. Test — once

The protocol allows **one** test evaluation. Narrow `FINAL` to the models you are reporting
before running this; thresholds fitted in section 4 are reused rather than refitted.

In [ ]:
# ONE test evaluation, at the end, on the models you are actually reporting.
# Narrow ONLY first -- every extra run here is another look at the frozen split.
FINAL = ["resnet34_finetune", "dilated-style-64-dihedral8"]

final_flags = [flag for name in FINAL for flag in ("--only", name)]
posthoc("--split", "both", "--final-test-evaluation",
        "--results", str(OUTPUT_ROOT / "final_test.csv"), *final_flags)

## 6. Read everything

In [ ]:
import pandas as pd

pd.set_option("display.width", 240)
for path in sorted(OUTPUT_ROOT.glob("*.csv")):
    frame = pd.read_csv(path)
    print(f"\n=== {path.name}  ({len(frame)} rows)")
    columns = [c for c in ("run", "split", "px", "tta", "thresholds", "macro_f1",
                           "ci_lower", "ci_upper", "balanced_accuracy",
                           "f1_Loc", "f1_Scratch", "f1_Edge-Loc", "f1_Near-full")
               if c in frame]
    display(frame.sort_values("macro_f1", ascending=False)[columns])

if HAS_DRIVE:
    import shutil
    for path in OUTPUT_ROOT.glob("*.csv"):
        shutil.copy2(path, DRIVE / f"posthoc_{path.name}")
    print("\ncopied to Drive")

## 7. Grad-CAM

One figure per class: the wafer, then where each model looked. A red title means that model
misclassified that wafer — worth keeping in the deck rather than cherry-picking hits.

Token models (ViT, Swin) are skipped: they have no final convolution whose layout matches
the image, so the same recipe would need attention rollout instead.

In [ ]:
# 2-4 checkpoints reads well on a slide. Full paths, not run names.
CAM_MODELS = [
    CHECKPOINTS_DIR / "v32-resnet34_finetune" / "best.pt",
    CHECKPOINTS_DIR / "v31-dilated_rotation_64" / "best.pt",
    CHECKPOINTS_DIR / "v27-convnext_style" / "best.pt",
]
CAM_CLASSES = ["Scratch", "Loc", "Edge-Loc", "Center"]
CAM_OUTPUT = REPO / "output/gradcam"

missing = [p for p in CAM_MODELS if not p.is_file()]
assert not missing, f"missing: {missing}\nRun the inspect cell to see what is on Drive."

command = [sys.executable, str(REPO / "scripts/gradcam.py"),
           "--classes", *CAM_CLASSES, "--output", str(CAM_OUTPUT)]
for path in CAM_MODELS:
    command += ["--checkpoint", str(path)]
print(" ".join(command), "\n")
subprocess.run(command, check=False)

## 8. Show them

In [ ]:
from IPython.display import Image, display

for path in sorted(CAM_OUTPUT.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

if HAS_DRIVE:
    target = DRIVE / "gradcam"
    target.mkdir(parents=True, exist_ok=True)
    for path in CAM_OUTPUT.glob("*.png"):
        shutil.copy2(path, target / path.name)
    print("copied to", target)